# Precision Agriculture Drone System: Unified Deep Learning & Edge Pipeline
### End-to-End Plant Disease Classification, Model Training (ResNet-9), Inference & Edge Deployment

This master notebook integrates and unifies all 3 project notebooks:
1. **`Download-dataset.ipynb`**: Kaggle API data ingestion and remote storage mounting.
2. **`Copy of plant-disease-classification-resnet-99-2.ipynb`**: ResNet-9 PyTorch architecture, OneCycleLR training, and 99.2% accuracy evaluation.
3. **`plant_desease_model_load.ipynb`**: Model serialization, bug-fixed weight loading, test inference, and drone prescription matrix integration.

---

## 1. Environment Setup & Data Ingestion (from `Download-dataset.ipynb`)
Supports running either in **Google Colab** with Google Drive/Kaggle or in a **Local Environment**.

In [ ]:
import os
import sys

# Check if running inside Google Colab
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    print("[Environment] Running in Google Colab. Mounting Google Drive...")
    from google.colab import drive
    drive.mount('/content/gdrive')
    
    # Optional Kaggle API credentials setup
    kaggle_dir = "/content/gdrive/My Drive/Kaggle"
    if os.path.exists(kaggle_dir):
        os.environ['KAGGLE_CONFIG_DIR'] = kaggle_dir
        print(f"[Kaggle] Credentials directory set to: {kaggle_dir}")
    
    # Path to New Plant Diseases Dataset on Google Drive or local /content
    data_dir = "/content/gdrive/My Drive/Kaggle/New Plant Diseases Dataset(Augmented)/New Plant Diseases Dataset(Augmented)"
else:
    print("[Environment] Running in Local / Edge Environment.")
    data_dir = "./data/New Plant Diseases Dataset(Augmented)"

print(f"Target data directory: {data_dir}")

## 2. Core Dependencies & GPU/CPU Device Setup

In [ ]:
import copy
import json
import time
import sqlite3
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
import torchvision.transforms as transforms
from torchvision.datasets import ImageFolder
from torchvision.utils import make_grid

# Set seeds for reproducibility
RANDOM_SEED = 42
torch.manual_seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)

# Hardware Acceleration Device Selection
def get_default_device():
    """Pick CUDA GPU if available, else Apple MPS or CPU."""
    if torch.cuda.is_available():
        return torch.device("cuda")
    elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")

def to_device(data, device):
    """Move tensor(s) to chosen device."""
    if isinstance(data, (list, tuple)):
        return [to_device(x, device) for x in data]
    return data.to(device, non_blocking=True)

class DeviceDataLoader:
    """Wrap a dataloader to move batches to target device."""
    def __init__(self, dl, device):
        self.dl = dl
        self.device = device

    def __iter__(self):
        for b in self.dl:
            yield to_device(b, self.device)

    def __len__(self):
        return len(self.dl)

device = get_default_device()
print(f"[PyTorch] Active Compute Device: {device}")

## 3. Dataset Classes & Data Loading Setup
The 38 standard plant disease classes from the *New Plant Diseases Dataset* covering staple crops such as Maize (Corn), Potato, and Tomato.

In [ ]:
# The 38 canonical plant disease classes
PLANT_CLASSES = [
    'Apple___Apple_scab',
    'Apple___Black_rot',
    'Apple___Cedar_apple_rust',
    'Apple___healthy',
    'Blueberry___healthy',
    'Cherry_(including_sour)___Powdery_mildew',
    'Cherry_(including_sour)___healthy',
    'Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot',
    'Corn_(maize)___Common_rust_',
    'Corn_(maize)___Northern_Leaf_Blight',
    'Corn_(maize)___healthy',
    'Grape___Black_rot',
    'Grape___Esca_(Black_Measles)',
    'Grape___Leaf_blight_(Isariopsis_Leaf_Spot)',
    'Grape___healthy',
    'Orange___Haunglongbing_(Citrus_greening)',
    'Peach___Bacterial_spot',
    'Peach___healthy',
    'Pepper,_bell___Bacterial_spot',
    'Pepper,_bell___healthy',
    'Potato___Early_blight',
    'Potato___Late_blight',
    'Potato___healthy',
    'Raspberry___healthy',
    'Soybean___healthy',
    'Squash___Powdery_mildew',
    'Strawberry___Leaf_scorch',
    'Strawberry___healthy',
    'Tomato___Bacterial_spot',
    'Tomato___Early_blight',
    'Tomato___Late_blight',
    'Tomato___Leaf_Mold',
    'Tomato___Septoria_leaf_spot',
    'Tomato___Spider_mites Two-spotted_spider_mite',
    'Tomato___Target_Spot',
    'Tomato___Tomato_Yellow_Leaf_Curl_Virus',
    'Tomato___Tomato_mosaic_virus',
    'Tomato___healthy'
]

print(f"Total Recognized Plant Disease Classes: {len(PLANT_CLASSES)}")

# Standard transform: Resizes images to (256, 256) and converts to PyTorch FloatTensor
transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.ToTensor()
])

train_dir = os.path.join(data_dir, "train") if os.path.exists(data_dir) else None
valid_dir = os.path.join(data_dir, "valid") if os.path.exists(data_dir) else None

if train_dir and os.path.exists(train_dir):
    print(f"[Data] Found training directory at: {train_dir}")
    train_ds = ImageFolder(train_dir, transform=transform)
    valid_ds = ImageFolder(valid_dir, transform=transform)
    
    BATCH_SIZE = 32
    train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
    valid_dl = DataLoader(valid_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
    
    train_dl = DeviceDataLoader(train_dl, device)
    valid_dl = DeviceDataLoader(valid_dl, device)
    print(f"[Data] Training Batches: {len(train_dl)}, Validation Batches: {len(valid_dl)}")
else:
    print("[Data] Dataset directory not present locally. Model will be instantiated in standalone/inference mode.")

## 4. ResNet-9 Deep Learning Architecture
Extracted from `Copy of plant-disease-classification-resnet-99-2.ipynb`.
A compact CNN with ~6.5M parameters designed for low-power edge inference on ARM companion computers.

In [ ]:
def accuracy(outputs, labels):
    """Compute multiclass accuracy."""
    _, preds = torch.max(outputs, dim=1)
    return torch.tensor(torch.sum(preds == labels).item() / len(preds))

class ImageClassificationBase(nn.Module):
    """Base training & validation metric logging."""
    def training_step(self, batch):
        images, labels = batch
        out = self(images)
        loss = F.cross_entropy(out, labels)
        return loss

    def validation_step(self, batch):
        images, labels = batch
        out = self(images)
        loss = F.cross_entropy(out, labels)
        acc = accuracy(out, labels)
        return {"val_loss": loss.detach(), "val_accuracy": acc}

    def validation_epoch_end(self, outputs):
        batch_losses = [x["val_loss"] for x in outputs]
        batch_accuracy = [x["val_accuracy"] for x in outputs]
        return {
            "val_loss": torch.stack(batch_losses).mean(),
            "val_accuracy": torch.stack(batch_accuracy).mean()
        }

    def epoch_end(self, epoch, result):
        print(f"Epoch [{epoch}], last_lr: {result['lrs'][-1]:.5f}, "
              f"train_loss: {result['train_loss']:.4f}, val_loss: {result['val_loss']:.4f}, "
              f"val_acc: {result['val_accuracy']:.4f}")

def ConvBlock(in_channels, out_channels, pool=False):
    """Conv2D + BatchNorm + ReLU with optional 4x Max Pooling."""
    layers = [
        nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
        nn.BatchNorm2d(out_channels),
        nn.ReLU(inplace=True)
    ]
    if pool:
        layers.append(nn.MaxPool2d(4))
    return nn.Sequential(*layers)

class ResNet9(ImageClassificationBase):
    """
    ResNet-9 Convolutional Neural Network:
    - Input: (3, 256, 256)
    - 4 ConvBlocks, 2 Residual Blocks (res1, res2)
    - Output: 38 Disease Classes
    """
    def __init__(self, in_channels=3, num_diseases=38):
        super().__init__()
        # Conv1: 3 -> 64
        self.conv1 = ConvBlock(in_channels, 64)
        # Conv2: 64 -> 128 (pool=True -> 64x64)
        self.conv2 = ConvBlock(64, 128, pool=True)
        # Res1: 128 -> 128 -> 128
        self.res1 = nn.Sequential(ConvBlock(128, 128), ConvBlock(128, 128))

        # Conv3: 128 -> 256 (pool=True -> 16x16)
        self.conv3 = ConvBlock(128, 256, pool=True)
        # Conv4: 256 -> 512 (pool=True -> 4x4)
        self.conv4 = ConvBlock(256, 512, pool=True)
        # Res2: 512 -> 512 -> 512
        self.res2 = nn.Sequential(ConvBlock(512, 512), ConvBlock(512, 512))

        # Classifier Head: MaxPool4 -> Flatten -> Linear(512, num_diseases)
        self.classifier = nn.Sequential(
            nn.MaxPool2d(4),
            nn.Flatten(),
            nn.Linear(512, num_diseases)
        )

    def forward(self, xb):
        out = self.conv1(xb)
        out = self.conv2(out)
        out = self.res1(out) + out
        out = self.conv3(out)
        out = self.conv4(out)
        out = self.res2(out) + out
        out = self.classifier(out)
        return out

# Instantiate model
model = ResNet9(3, len(PLANT_CLASSES))
model.to(device)
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"[Model] ResNet-9 successfully initialized with {total_params:,} trainable parameters.")

## 5. OneCycleLR Training & Evaluation Harness
Trains ResNet-9 using the OneCycle learning rate policy to achieve **99.2% validation accuracy**.

In [ ]:
@torch.no_grad()
def evaluate(model, val_loader):
    model.eval()
    outputs = [model.validation_step(batch) for batch in val_loader]
    return model.validation_epoch_end(outputs)

def get_lr(optimizer):
    for param_group in optimizer.param_groups:
        return param_group['lr']

def fit_OneCycle(epochs, max_lr, model, train_loader, val_loader, weight_decay=0, grad_clip=None, opt_func=torch.optim.SGD):
    torch.cuda.empty_cache()
    history = []
    optimizer = opt_func(model.parameters(), max_lr, weight_decay=weight_decay)
    sched = torch.optim.lr_scheduler.OneCycleLR(optimizer, max_lr, epochs=epochs, steps_per_epoch=len(train_loader))

    for epoch in range(epochs):
        model.train()
        train_losses = []
        lrs = []
        for batch in train_loader:
            loss = model.training_step(batch)
            train_losses.append(loss)
            loss.backward()

            if grad_clip:
                nn.utils.clip_grad_value_(model.parameters(), grad_clip)

            optimizer.step()
            optimizer.zero_grad()

            lrs.append(get_lr(optimizer))
            sched.step()

        result = evaluate(model, val_loader)
        result['train_loss'] = torch.stack(train_losses).mean().item()
        result['lrs'] = lrs
        model.epoch_end(epoch, result)
        history.append(result)
    return history

# Note: In Google Colab or local training environment with the dataset present, execute:
# epochs = 2
# max_lr = 0.01
# history = fit_OneCycle(epochs, max_lr, model, train_dl, valid_dl, grad_clip=0.1, weight_decay=1e-4, opt_func=torch.optim.Adam)

## 6. Model Serialization & Export (from `Copy of plant-disease...`)
Saves both the PyTorch `state_dict` and the class label metadata list for the drone edge engine.

In [ ]:
STATE_PATH = './plant-disease-model.pth'
COMPLETE_PATH = './plant-disease-model-complete.pth'
CLASSES_PATH = './plant_classes.json'

# 1. Save state_dict
torch.save(model.state_dict(), STATE_PATH)
print(f"[Export] Saved state_dict to: {STATE_PATH}")

# 2. Save complete model
torch.save(model, COMPLETE_PATH)
print(f"[Export] Saved complete model graph to: {COMPLETE_PATH}")

# 3. Export class labels JSON
with open(CLASSES_PATH, 'w') as f:
    json.dump(PLANT_CLASSES, f, indent=2)
print(f"[Export] Saved class definitions to: {CLASSES_PATH}")

## Download Trained ResNet-9 Model Weights


In [ ]:
import os

# Download the trained ResNet-9 model weights (26,397,109 bytes) from Colab to your local PC
MODEL_FILE = "plant-disease-model.pth"

if os.path.exists(MODEL_FILE):
    print(f"Found trained ResNet-9 weights: {MODEL_FILE} ({os.path.getsize(MODEL_FILE):,} bytes).")
    try:
        from google.colab import files
        print("Downloading ResNet-9 model to your computer via browser...")
        files.download(MODEL_FILE)
        if os.path.exists("plant_classes.json"):
            files.download("plant_classes.json")
        print("Download triggered! Check your browser Downloads folder.")
    except ImportError:
        print(f"Running locally. ResNet-9 model ready at: {os.path.abspath(MODEL_FILE)}")
else:
    print(f"'{MODEL_FILE}' not found. Run Section 5 & 6 above to train and save the ResNet-9 model.")


## 7. Model Reloading & Verification (Fixing the Bug in `plant_desease_model_load.ipynb`)
In the original `plant_desease_model_load.ipynb`, cell 13 contained a critical bug:
```python
# BUG in original notebook:
model = ResNet9(3, len(train.classes))
torch.save(model.state_dict(), PATH)  # <-- OVERWROTE TRAINED WEIGHTS WITH UNTRAINED MODEL!
model.load_state_dict(torch.load(PATH))
```
Below is the **corrected, robust loading routine**.

In [ ]:
def safe_load_model(weights_path, num_classes=38, device=None):
    """
    Robust loader that safely loads weights without accidental overwriting.
    """
    if device is None:
        device = get_default_device()

    loaded_model = ResNet9(in_channels=3, num_diseases=num_classes)
    loaded_model.to(device)

    if os.path.exists(weights_path):
        checkpoint = torch.load(weights_path, map_location=device)
        if isinstance(checkpoint, dict) and 'state_dict' in checkpoint:
            loaded_model.load_state_dict(checkpoint['state_dict'])
        elif isinstance(checkpoint, dict):
            loaded_model.load_state_dict(checkpoint)
        elif isinstance(checkpoint, nn.Module):
            loaded_model = checkpoint
            loaded_model.to(device)
        print(f"[Success] Loaded model weights from: {weights_path}")
    else:
        print(f"[Warning] Weights file '{weights_path}' not found. Using initialized model.")

    loaded_model.eval()
    return loaded_model

# Test safe loading
eval_model = safe_load_model(STATE_PATH, num_classes=len(PLANT_CLASSES), device=device)

## 8. Real-Time Single Frame & Batch Inference Engine
Runs disease prediction, calculates softmax confidence as severity percentage, and visualizes predictions.

In [ ]:
def predict_image(img_tensor, model, classes=PLANT_CLASSES, device=None):
    """
    Evaluate image tensor and return predicted class and confidence percentage.
    """
    if device is None:
        device = get_default_device()
        
    # Add batch dimension: (1, 3, 256, 256)
    xb = to_device(img_tensor.unsqueeze(0), device)
    with torch.no_grad():
        yb = model(xb)
        probs = torch.softmax(yb, dim=1)
        confidence, preds = torch.max(probs, dim=1)
        
    pred_idx = preds[0].item()
    pred_class = classes[pred_idx]
    confidence_pct = float(confidence.item()) * 100.0
    return pred_class, confidence_pct

# Test with synthetic canopy tensor
sample_tensor = torch.randn(3, 256, 256)
predicted_disease, severity = predict_image(sample_tensor, eval_model, PLANT_CLASSES, device)
print(f"[Inference Test] Predicted: {predicted_disease} | Confidence/Severity: {severity:.2f}%")

## 9. Edge Drone Deployment & SQLite Prescription Integration
Connects model predictions directly to the drone's offline relational database (`crop_health_edge.db`).
Converts disease classifications into actionable, site-specific pesticide prescriptions.

In [9]:
# Connect to the drone's offline relational database
import os, sqlite3

# Search common candidate locations for crop_health_edge.db
possible_paths = [
    "crop_health_edge.db",
    os.path.join(os.getcwd(), "crop_health_edge.db"),
    "/home/kami/Desktop/codebase/gomi/crop_health_edge.db"
]

DB_PATH = "crop_health_edge.db"
for p in possible_paths:
    if os.path.exists(p):
        DB_PATH = p
        break

def ensure_database_ready(db_path=DB_PATH):
    """Auto-initialize and seed SQLite database if not present in environment."""
    if not os.path.exists(db_path) or os.path.getsize(db_path) == 0:
        try:
            from db_init import create_pest_database, seed_agronomic_records
            create_pest_database(db_path)
            seed_agronomic_records(db_path)
            print(f"[DB] Auto-seeded complete agronomic matrix into: {db_path}")
        except Exception:
            # Fallback self-contained inline table initialization
            conn = sqlite3.connect(db_path)
            c = conn.cursor()
            c.execute("CREATE TABLE IF NOT EXISTS Pest_Signature_Table (pest_id INTEGER PRIMARY KEY AUTOINCREMENT, pest_name TEXT NOT NULL UNIQUE, crop_type TEXT, keypoints_blob BLOB, descriptors_blob BLOB, severity_class TEXT);")
            c.execute("CREATE TABLE IF NOT EXISTS Pesticide_Matrix_Table (matrix_id INTEGER PRIMARY KEY AUTOINCREMENT, target_pest_id INTEGER, min_severity_pct REAL, max_severity_pct REAL, recommended_chemical TEXT, dosage_ml_per_litre REAL, active_ingredient TEXT, safety_interval_days INTEGER);")
            c.execute("INSERT OR IGNORE INTO Pest_Signature_Table (pest_id, pest_name, crop_type, keypoints_blob, descriptors_blob, severity_class) VALUES (1, 'Corn_(maize)___Common_rust_', 'Corn_(maize)', x'00', x'00', 'Fungal');")
            c.execute("INSERT OR IGNORE INTO Pesticide_Matrix_Table (target_pest_id, min_severity_pct, max_severity_pct, recommended_chemical, dosage_ml_per_litre, active_ingredient) VALUES (1, 15.0, 45.0, 'Mancozeb 75% WP', 2.5, 'Dithiocarbamate');")
            c.execute("INSERT OR IGNORE INTO Pesticide_Matrix_Table (target_pest_id, min_severity_pct, max_severity_pct, recommended_chemical, dosage_ml_per_litre, active_ingredient) VALUES (1, 45.1, 100.0, 'Azoxystrobin 23% SC', 1.0, 'Strobilurin');")
            conn.commit()
            conn.close()
            print(f"[DB] Initialized fallback pesticide matrix in: {db_path}")

def query_drone_prescription(pest_name, severity_pct, db_path=DB_PATH):
    """Query offline SQLite Pesticide_Matrix_Table based on model classification."""
    ensure_database_ready(db_path)

    if "healthy" in pest_name.lower() or severity_pct < 10.0:
        return {
            "pest": pest_name,
            "pest_name": pest_name,
            "severity": float(severity_pct),
            "severity_pct": float(severity_pct),
            "action": "NO_ACTION",
            "chemical": "None (Canopy Healthy)",
            "recommended_chemical": "None (Canopy Healthy)",
            "dosage_ml_per_l": 0.0,
            "dosage_ml_per_litre": 0.0
        }

    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()
    cursor.execute("""
    SELECT p.pest_name, m.recommended_chemical, m.dosage_ml_per_litre, m.active_ingredient
    FROM Pest_Signature_Table p
    JOIN Pesticide_Matrix_Table m ON p.pest_id = m.target_pest_id
    WHERE p.pest_name = ?
      AND ? >= m.min_severity_pct AND ? <= m.max_severity_pct
    LIMIT 1;
    """, (pest_name, severity_pct, severity_pct))
    
    row = cursor.fetchone()
    conn.close()
    
    if row:
        return {
            "pest": row[0],
            "pest_name": row[0],
            "severity": float(severity_pct),
            "severity_pct": float(severity_pct),
            "chemical": row[1],
            "recommended_chemical": row[1],
            "dosage_ml_per_l": float(row[2]),
            "dosage_ml_per_litre": float(row[2]),
            "active_ingredient": row[3],
            "action": "SPOT_SPRAY"
        }
    return {
        "pest": pest_name,
        "pest_name": pest_name,
        "severity": float(severity_pct),
        "severity_pct": float(severity_pct),
        "chemical": "Broad-Spectrum Agronomic Protection (Mancozeb 75% WP)",
        "recommended_chemical": "Broad-Spectrum Agronomic Protection (Mancozeb 75% WP)",
        "dosage_ml_per_l": 2.0,
        "dosage_ml_per_litre": 2.0,
        "active_ingredient": "Dithiocarbamate Protective Complex",
        "action": "SPOT_TREAT"
    }

# Demonstration: Simulating a Corn Common Rust detection from the model
demo_pest = "Corn_(maize)___Common_rust_"
demo_severity = 48.5
prescription = query_drone_prescription(demo_pest, demo_severity)

print("="*70)
print("AUTONOMOUS DRONE PRESCRIPTION ENGINE OUTPUT")
print("="*70)
print(f"Target Crop Disease:    {prescription['pest']}")
print(f"Severity Percentage:    {prescription['severity']}%")
print(f"Action Protocol:        {prescription['action']}")
print(f"Prescribed Chemical:    {prescription['chemical']}")
print(f"Recommended Dosage:     {prescription['dosage_ml_per_l']} ml/Litre")
print("="*70)


AUTONOMOUS DRONE PRESCRIPTION ENGINE OUTPUT
Target Crop Disease:    Corn_(maize)___Common_rust_
Severity Percentage:    48.5%
Action Protocol:        SPOT_SPRAY
Prescribed Chemical:    Azoxystrobin 23% SC
Recommended Dosage:     1.0 ml/Litre


## 10. Summary & Production Edge Deployment
To deploy this model onto the drone's **Raspberry Pi 4 Companion Computer**:
1. Keep `plant-disease-model.pth` and `plant_classes.json` in the drone workspace.
2. The drone's real-time loop ([`mission_runner.py`](file:///home/kami/Desktop/codebase/gomi/mission_runner.py)) automatically loads [`model.py`](file:///home/kami/Desktop/codebase/gomi/model.py) and [`vision_engine.py`](file:///home/kami/Desktop/codebase/gomi/vision_engine.py).
3. Telemetry coordinates from the Pixhawk 4 autopilot are linked to every detection and saved to `field_prescription_log.csv` and `field_prescription_map.geojson`.